In [ ]:
import Pkg
Pkg.activate("/Users/users/boyea/DATASERVER/julia/")

In [ ]:
using CairoMakie

In [ ]:
using HDF5

In [ ]:
using Random, Statistics

In [ ]:
using Specje

In [ ]:
using AdvancedHMC, AbstractMCMC


In [ ]:
h5f = h5open("./example_fits/39627867289552627/spectrum.hdf5")
wave = h5f["wavelength"][:]
flux = h5f["flux"][:]
uncertainty = h5f["uncertainty"][:]
disp = h5f["wave_dispersion"][:]

redshift = attrs(h5f)["redshift"]

close(h5f)

In [ ]:
lib = read_fsps_hdf5("fsps_ssp.h5")

In [ ]:
spec = Spectrum(
    wave / (1 + redshift),
    flux, 
    uncertainty,
    good = isfinite.(uncertainty),
    lsf_fwhm = disp
    )

In [ ]:
mwave = model_grid(spec.wave; oversample=2)    # log grid covering every native pixel
P     = pixel_projection(mwave, spec.wave)     # sparse (native pixels × model pixels)
vs    = velscale(mwave)                        # km/s per model pixel

In [ ]:
minimum(mwave)

In [ ]:
tlib = rebin_log(lib; velscale=vs, λmin=3000, λmax=11_000)
# tlib = normalize_window(tlib; window=(4500.0, 4600.0))

In [ ]:
lsf  = GaussianLSF(spec, mwave)              # the data's LSF, interpolated onto the model grid
m    = match_resolution_auto(tlib, mwave, lsf)
grid = kinematics_grid(tlib.wave, mwave)     # FFT geometry linking the two grids

In [ ]:
star = StellarComponent(grid, m.templates; σ_min=10.0, σ_max=600.0, vmax=1000.0)
gas  = GasComponent(mwave,
                    [GasGroup("balmer", Multiplet("Hbeta", 4862.683),
                                        Multiplet("Hgamma", 4341.692))],
                    lsf; σ_min=5.0, vmax=800.0)
poly = PolynomialComponent(length(mwave), 2)

model = SpectralModel(grid, star, gas, poly; projection=P)   # θ = (v★, σ★, v_gas, σ_gas)
nparams(model), ncolumns(model)                              # (4, 16)
npix(model)                                                  # native pixels

In [ ]:
fit = fit_kinematics(model, spec, [0.0, 150.0, 0.0, 100.0])
fit.θ, reduced_chi2(fit), fit.converged

# Bayesian


In [ ]:
fd = FitData(spec)
bm = BayesModel(model, fd, fit.θ; fit_noise=true)


In [ ]:
initial_free(bm, fit.θ)

In [ ]:
u_map = map_estimate(bm, initial_free(bm, fit.θ)).u

In [ ]:
p0 = initial_free(bm, fit.θ)

In [ ]:
samples = AbstractMCMC.sample(Xoshiro(1), AbstractMCMC.LogDensityModel(bm), NUTS(0.8), 2000;
                              n_adapts=1000, initial_params=p0, progress=true)

In [ ]:
samples

In [ ]:
chain = sample_ensemble(Xoshiro(1), bm, initial_free(bm, fit.θ); nsteps=2000)
acceptance_fraction(chain), autocorr_time(chain)
U = flatten_chain(chain; burn=500)
Θ = reduce(hcat, [to_physical(bm, u).θ for u in eachcol(U)])
mean(Θ; dims=2), std(Θ; dims=2)

In [ ]:
wp = weight_posterior(bm, U[:, end])
draws = sample_weights(Xoshiro(2), wp, 200; positive=true)   # (ncolumns, 200)